# Attention Optimizer Results
Pull run histories from W&B by default, with a local JSONL fallback.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import wandb

LOGS_DIR = Path('../logs')
SMOOTH_WINDOW = 50
WANDB_ENTITY = os.environ.get('WANDB_ENTITY')
WANDB_PROJECT = os.environ.get('WANDB_PROJECT', 'attn-optimizer')
USE_WANDB = WANDB_ENTITY is not None

GROUPS = {
    'Baselines': ['BASE-SGD', 'BASE-ADAM', 'BASE-ADAMW', 'BASE-MUON'],
    'Pure Attention': ['ATTN-PURE-8-TRAIN', 'ATTN-PURE-16-TRAIN'],
    'Gated Attention': ['ATTN-GATED-8-TRAIN', 'ATTN-GATED-16-TRAIN'],
}

LABELS = {
    'BASE-SGD': 'SGD',
    'BASE-ADAM': 'Adam',
    'BASE-ADAMW': 'AdamW',
    'BASE-MUON': 'Muon',
    'ATTN-PURE-8-TRAIN': 'PureAttn-8',
    'ATTN-PURE-16-TRAIN': 'PureAttn-16',
    'ATTN-GATED-8-TRAIN': 'GatedAttn-8',
    'ATTN-GATED-16-TRAIN': 'GatedAttn-16',
}

In [ ]:
def load_metrics_local(run_id):
    path = LOGS_DIR / run_id / 'metrics.jsonl'
    if not path.exists():
        return None, None
    steps, losses = [], []
    with open(path) as f:
        for line in f:
            d = json.loads(line)
            steps.append(d['step'])
            losses.append(d['loss'])
    return np.array(steps), np.array(losses)

def load_metrics_wandb(run_id, samples=2000):
    api = wandb.Api()
    runs = api.runs(
        f'{WANDB_ENTITY}/{WANDB_PROJECT}',
        filters={'display_name': run_id},
    )
    if len(runs) == 0:
        return None, None
    run = runs[0]
    history = run.history(keys=['loss'], x_axis='_step', samples=samples, pandas=True)
    history = history.dropna(subset=['loss'])
    if history.empty:
        return None, None
    steps = history['_step'].to_numpy()
    losses = history['loss'].to_numpy()
    return steps, losses

def load_metrics(run_id):
    if USE_WANDB:
        steps, losses = load_metrics_wandb(run_id)
        if steps is not None:
            return steps, losses
    return load_metrics_local(run_id)

def smooth(x, w):
    if len(x) < w:
        return x
    kernel = np.ones(w) / w
    return np.convolve(x, kernel, mode='valid')

all_runs = {rid: load_metrics(rid) for group in GROUPS.values() for rid in group}
print('Using W&B:' if USE_WANDB else 'Using local logs only.', WANDB_ENTITY or '')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.tab20.colors

for i, (run_id, (steps, losses)) in enumerate(all_runs.items()):
    if steps is None:
        continue
    s_loss = smooth(losses, SMOOTH_WINDOW)
    s_steps = steps if len(losses) < SMOOTH_WINDOW else steps[SMOOTH_WINDOW - 1:]
    ax.plot(s_steps, s_loss, label=LABELS.get(run_id, run_id), color=colors[i], lw=1.5)

ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Training Loss — All Runs')
ax.legend(fontsize=8, ncol=3)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes = axes.flatten()

for ax, (group_name, run_ids) in zip(axes, GROUPS.items()):
    for run_id in run_ids:
        steps, losses = all_runs[run_id]
        if steps is None:
            continue
        s_loss = smooth(losses, SMOOTH_WINDOW)
        s_steps = steps if len(losses) < SMOOTH_WINDOW else steps[SMOOTH_WINDOW - 1:]
        ax.plot(s_steps, s_loss, label=LABELS.get(run_id, run_id), lw=1.8)
    ax.set_title(group_name)
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print(f"{'Run ID':<20}  {'Optimizer':<22}  {'Model':<12}  {'Final Loss':>10}")
print('-' * 72)
for run_id, (steps, losses) in all_runs.items():
    if steps is None:
        print(f"{run_id:<20}  {'(not found)':<22}")
        continue
    final = float(np.mean(losses[-min(len(losses), 100):]))
    print(f"{run_id:<20}  {LABELS.get(run_id, run_id):<22}  {'gpt':<12}  {final:>10.4f}")